# SQL Database Agent @ `LangGraph`

## Creating & Populating Database

In [52]:
import sqlite3

In [53]:
# Connect to SQLite database

connection = sqlite3.connect('mydb.db')

In [54]:
connection

In [55]:
# Creating table(s)

table_create_query_1 = """
CREATE TABLE IF NOT EXISTS employees (
    emp_id INTEGER PRIMARY KEY,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    prhire_date TEXT NOT NULL,
    salary INTEGER NOT NULL
);
"""

In [56]:
table_create_query_2 = """
CREATE TABLE IF NOT EXISTS customers (
  customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
  first_name TEXT NOT NULL,
  last_name TEXT NOT NULL,
  email TEXT UNIQUE NOT NULL,
  phone TEXT
);
"""

In [57]:
table_create_query_3 = """
CREATE TABLE IF NOT EXISTS orders (
    order_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    amount REAL NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers (customer_id)
);"""

In [58]:
# Creating cursor object

cursor = connection.cursor()

In [59]:
# Passing table creation queries to the database

cursor.execute(table_create_query_1)
cursor.execute(table_create_query_2)
cursor.execute(table_create_query_3)

In [60]:
# Dummy Data for Employees
employees_data = [
    (1, 'Alice', 'Johnson', 'alice.j@example.com', '2023-01-15', 75000),
    (2, 'Bob', 'Smith', 'bob.s@example.com', '2023-03-20', 68000),
    (3, 'Charlie', 'Davis', 'charlie.d@example.com', '2022-11-05', 82000),
    (4, 'Diana', 'Prince', 'diana.p@example.com', '2024-02-10', 95000),
    (5, 'Edward', 'Norton', 'edward.n@example.com', '2023-06-30', 70000)
]

# Dummy Data for Customers
customers_data = [
    ('John', 'Doe', 'john.doe@email.com', '555-0101'),
    ('Jane', 'Smith', 'jane.smith@email.com', '555-0102'),
    ('Michael', 'Brown', 'michael.b@email.com', '555-0103'),
    ('Emily', 'Davis', 'emily.d@email.com', '555-0104'),
    ('Chris', 'Wilson', 'chris.w@email.com', '555-0105')
]

# Dummy Data for Orders
import datetime
orders_data = [
    (1, datetime.date(2024, 1, 10).isoformat(), 150.50),
    (2, datetime.date(2024, 1, 12).isoformat(), 200.00),
    (1, datetime.date(2024, 2, 5).isoformat(), 45.75),
    (3, datetime.date(2024, 2, 14).isoformat(), 320.00),
    (4, datetime.date(2024, 3, 1).isoformat(), 12.99),
    (5, datetime.date(2024, 3, 5).isoformat(), 89.50),
    (2, datetime.date(2024, 3, 10).isoformat(), 110.25),
    (3, datetime.date(2024, 3, 15).isoformat(), 55.00),
    (1, datetime.date(2024, 3, 20).isoformat(), 250.00),
    (4, datetime.date(2024, 3, 22).isoformat(), 75.60)
]

cursor = connection.cursor()

# Insert Employees
cursor.executemany("INSERT OR IGNORE INTO employees (emp_id, first_name, last_name, email, prhire_date, salary) VALUES (?, ?, ?, ?, ?, ?)", employees_data)

# Insert Customers (using NULL for auto-increment ID)
cursor.executemany("INSERT OR IGNORE INTO customers (first_name, last_name, email, phone) VALUES (?, ?, ?, ?)", customers_data)

# Insert Orders
cursor.executemany("INSERT OR IGNORE INTO orders (customer_id, order_date, amount) VALUES (?, ?, ?)", orders_data)

connection.commit()
print("Dummy data populated successfully!")


Dummy data populated successfully!


In [61]:
# Retrieving data from table "orders"
cursor.execute("SELECT * FROM orders;")

In [62]:
for i in cursor.fetchall(): # Rows from table "orders"
    print(i)

(1, 1, '2024-01-10', 150.5)
(2, 2, '2024-01-12', 200.0)
(3, 1, '2024-02-05', 45.75)
(4, 3, '2024-02-14', 320.0)
(5, 4, '2024-03-01', 12.99)
(6, 5, '2024-03-05', 89.5)
(7, 2, '2024-03-10', 110.25)
(8, 3, '2024-03-15', 55.0)
(9, 1, '2024-03-20', 250.0)
(10, 4, '2024-03-22', 75.6)


<br>
<br>

# Creating `Agent`

In [63]:
from langchain_community.utilities import SQLDatabase

In [64]:
db = SQLDatabase.from_uri("sqlite:///mydb.db") # SQLite database

In [65]:
db

In [66]:
db.dialect # Version of SQL

'sqlite'

In [67]:
db.get_usable_table_names() # List of the tables created

['customers', 'employees', 'orders']

In [68]:
from langchain_groq import ChatGroq

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

llm = ChatGroq(model="llama-3.1-8b-instant")

In [69]:
llm.invoke("Hello, how are you?") # Invoking the LLM

AIMessage(content="I'm doing well, thank you for asking. I'm a large language model, so I don't have feelings or emotions like humans do, but I'm here and ready to help with any questions or topics you'd like to discuss. How about you? How's your day going?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 41, 'total_tokens': 100, 'completion_time': 0.065913867, 'completion_tokens_details': None, 'prompt_time': 0.007093031, 'prompt_tokens_details': None, 'queue_time': 0.068421763, 'total_time': 0.073006898}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_d317489708', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cb149-640a-74f3-8d9f-bd9268aa70a0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 59, 'total_tokens': 100})

In [70]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

In [71]:
# Creating the toolkit for all the available tools

toolkit = SQLDatabaseToolkit(db=db, llm=llm)

In [72]:
toolkit.get_tools()

[QuerySQLDatabaseTool(description="Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.", db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x1078761b0>),
 InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x1078761b0>),
 ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x1078761b0>),
 QuerySQLCheckerTool(description='Use this tool to double check if your 

In [73]:
tools = toolkit.get_tools()

In [74]:
for tool in tools:
  print(tool.name)

sql_db_query
sql_db_schema
sql_db_list_tables
sql_db_query_checker


In [75]:
list_tables_tool = next((tool for tool in tools if tool.name == "sql_db_list_tables"), None)

In [76]:
list_tables_tool

ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x1078761b0>)

In [77]:
# Fetching table schema tool

get_schema_tool = next((tool for tool in tools if tool.name == "sql_db_schema"), None)

In [78]:
get_schema_tool

InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x1078761b0>)

In [79]:
list_tables_tool.invoke("")

'customers, employees, orders'

In [ ]:
print(get_schema_tool.invoke("customers")) # Getting the schema of the table "custoers" + first few rows


CREATE TABLE customers (
	customer_id INTEGER, 
	first_name TEXT NOT NULL, 
	last_name TEXT NOT NULL, 
	email TEXT NOT NULL, 
	phone TEXT, 
	PRIMARY KEY (customer_id), 
	UNIQUE (email)
)

/*
3 rows from customers table:
customer_id	first_name	last_name	email	phone
1	John	Doe	john.doe@email.com	555-0101
2	Jane	Smith	jane.smith@email.com	555-0102
3	Michael	Brown	michael.b@email.com	555-0103
*/
